# MLIR 编译器主线 · 第 4/8 课：Pass、PatternRewriter 与贪心重写

> 状态：**参考答案版**  
> 本仓库采用逐课通过制。本课未通过前，不应直接进入下一课。

## 本课目标与完成标准

学完后你应能：编写局部 rewrite，解释 benefit、收敛与 `PatternRewriter` 修改纪律。

通过必须同时满足：

- 独立补齐本课唯一的代码填空题，并通过给定检查；
- 三个问答题均说明因果链，而不是只报术语；
- 能指出至少一个正确性边界和一个性能取舍；
- 总分不低于 8/10，且没有一票否决级概念错误。

## 前置关系

- 课程前置：编译原理基础、C++ 阅读能力
- 本课在路线中的作用：Pass 决定遍历/阶段，RewritePattern 匹配局部 IR；PatternRewriter 记录替换，维护 def-use 与监听器不变量。

## 核心心智模型

### 1. 它是什么，解决什么问题

Pass 决定遍历/阶段，RewritePattern 匹配局部 IR；PatternRewriter 记录替换，维护 def-use 与监听器不变量。

### 2. 它如何工作

pattern 检查根 op 与额外条件，创建新 op 后 `replaceOp`；greedy driver 反复应用直到固定点或达到限制。

### 3. 正确性条件与常见误区

不能绕过 rewriter 直接 erase/replace 被管理的 op；A→B 与 B→A 会振荡，pattern 必须有单调度量。

### 4. 性能与工程取舍

小 pattern 易组合，但贪心结果受 benefit 与可匹配顺序影响；复杂全局优化可能需要专用算法。

## 具体演示

`x+0→x` 减少 operation 数，是明显单调的 canonicalization；`x→x+0` 则破坏终止。

请在阅读后先合上这一节，用自己的语言复述“输入状态 → 中间状态 → 输出状态”，再做练习。

## 实践任务：唯一代码填空题

补齐 `x + 0 -> x` 的替换动作。

规则：只能修改 `TODO`/`______` 所在位置；不要删除断言或放宽误差。代码注释说明了每个边界条件。

In [ ]:
// C++ 片段：假设已确认 rhs 是整数 0。
LogicalResult matchAndRewrite(arith::AddIOp op,
                              PatternRewriter &rewriter) const override {
  Value x = op.getLhs();
  rewriter.______(op, ______);
  return success();
}


### 检查方法

在完整插件环境运行 pass 两次，第二次 IR 应不再变化；当前无构建环境时做 API/def-use 静态审查。

提交时请给出：补齐后的代码、实际运行输出（环境不可用时注明“仅静态审查”）以及对失败用例的解释。

### Q1

不要背定义：请从输入、状态变化和输出三个阶段解释“Pass、PatternRewriter 与贪心重写”的工作机制。

**你的答案：**


### Q2

直接调用 `op.erase()` 为什么可能破坏 rewrite driver？

**你的答案：**


### Q3

两个都正确但互相逆向的 canonicalization 如何避免振荡？

**你的答案：**


## 评分与通过规则

- 代码 4 分：正常输入 2 分，边界输入 1 分，解释实现 1 分；
- Q1～Q3 各 2 分；
- 一票否决：结果碰巧正确但核心因果链错误、删除边界检查、把未运行结果说成实测。

需要提示时按四级机制请求：概念区域 → 具体方向 → 关键局部 → 完整答案。

## 参考答案（仅 answer 分支）

先完成题目再核对。即使代码一致，也要能解释关键步骤，并尝试更换一个输入规模。

In [ ]:
LogicalResult matchAndRewrite(arith::AddIOp op,
                              PatternRewriter &rewriter) const override {
  Value x = op.getLhs();
  rewriter.replaceOp(op, x);
  return success();
}


### Q1 参考答案

pattern 检查根 op 与额外条件，创建新 op 后 `replaceOp`；greedy driver 反复应用直到固定点或达到限制。

### Q2 参考答案

判断时先检查本课不变量：不能绕过 rewriter 直接 erase/replace 被管理的 op；A→B 与 B→A 会振荡，pattern 必须有单调度量。  若不成立，最终数值或系统状态即使暂时正常也不可信。

### Q3 参考答案

迁移时先保证正确性，再比较代价。这里的核心取舍是：小 pattern 易组合，但贪心结果受 benefit 与可匹配顺序影响；复杂全局优化可能需要专用算法。

## 参考资料

- [Pattern Rewriter](https://mlir.llvm.org/docs/PatternRewriter/)
- [MLIR Toy Tutorial](https://mlir.llvm.org/docs/Tutorials/Toy/)

资料用于建立事实基线；面试回答仍需用自己的语言组织。